In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [2]:
import numpy as np
import pyreadr
from pathlib import Path
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, diags
import geopandas as gpd

# ================================================================
# 0. CONFIG
# ================================================================

BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# 1. LOAD DATA (NON-ISOLATED ONLY)
# ================================================================

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

y_full = snow.iloc[:, 2:].to_numpy()
coords_full = snow.iloc[:, :2].to_numpy()

# drop isolated
y = np.delete(y_full, no_nbs, axis=0)
coords = np.delete(coords_full, no_nbs, axis=0)

S, T = y.shape
print(f"S = {S}, T = {T}")

# ================================================================
# 2. BUILD ADJACENCY MATRIX (SAME AS MODEL)
# ================================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

# rotation (as in model)
dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
rotated = (xy @ R.T) / 1e6

Distances = squareform(pdist(rotated))
W = (Distances <= 0.22).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

deg = np.array(W.sum(axis=1)).flatten()
assert np.all(deg > 0)

# ================================================================
# 3. OBSERVED STATISTIC
# T_obs[t] = sum_i sum_j W_ij * y_{j,t}
# t = 2,...,T  (Python: 1,...,T-1)
# ================================================================

T_obs = np.zeros(T-1)

for t in range(1, T):
    neigh_sum = W @ y[:, t]
    T_obs[t-1] = neigh_sum.sum()

# ================================================================
# 4. LOAD POSTERIOR SAMPLES
# ================================================================

bym01 = np.load(BASE_DIR / "bym01_noIso_final.npz")["all_theta"]
bym10 = np.load(BASE_DIR / "bym10_noIso_final.npz")["all_theta"]

ind01_full = np.load(BASE_DIR / "ind01.npz")["all_theta"]
ind10_full = np.load(BASE_DIR / "ind10.npz")["all_theta"]

M = bym01.shape[1]
print(f"Posterior samples M = {M}")

# extract non-isolated IID parameters
S_full = y_full.shape[0]
K_iid = 4

ind01 = np.zeros((K_iid*S, M))
ind10 = np.zeros((K_iid*S, M))

for k in range(K_iid):
    ind01[k*S:(k+1)*S, :] = ind01_full[k*S_full : k*S_full + S, :]
    ind10[k*S:(k+1)*S, :] = ind10_full[k*S_full : k*S_full + S, :]

# ================================================================
# 5. BUILD COVARIATES (MATCH MCMC EXACTLY)
# ================================================================

t_raw = np.arange(1, T+1)
t_trend = (t_raw - t_raw.mean()) / t_raw.std()   # CRITICAL

cov_bym = np.column_stack([
    np.ones(T), np.ones(T),
    np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
    t_trend, t_trend
])   # (T, 8)

cov_iid = np.column_stack([
    np.ones(T),
    np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period),
    t_raw
])   # (T, 4)

# ================================================================
# 6. PPC — BYM (FORWARD SIMULATION)
# ================================================================

T_rep_bym = np.zeros((M, T-1))

print("\nRunning PPC (BYM)...")

for m in tqdm(range(M), desc="BYM neighbor PPC"):

    y_rep = np.zeros((S, T), dtype=int)
    y_rep[:, 0] = y[:, 0]

    for t in range(1, T):

        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(8):
            eta01 += cov_bym[t, k] * bym01[k*S:(k+1)*S, m]
            eta10 += cov_bym[t, k] * bym10[k*S:(k+1)*S, m]

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))

        prob = np.where(
            y_rep[:, t-1] == 0,
            p01,
            1 - p10
        )

        y_rep[:, t] = np.random.binomial(1, prob)

        neigh_sum = W @ y_rep[:, t]
        T_rep_bym[m, t-1] = neigh_sum.sum()

# ================================================================
# 7. PPC — IID (FORWARD SIMULATION)
# ================================================================

T_rep_iid = np.zeros((M, T-1))

print("\nRunning PPC (IID)...")

for m in tqdm(range(M), desc="IID neighbor PPC"):

    y_rep = np.zeros((S, T), dtype=int)
    y_rep[:, 0] = y[:, 0]

    for t in range(1, T):

        eta01 = np.zeros(S)
        eta10 = np.zeros(S)

        for k in range(4):
            eta01 += cov_iid[t, k] * ind01[k*S:(k+1)*S, m]
            eta10 += cov_iid[t, k] * ind10[k*S:(k+1)*S, m]

        p01 = 1 / (1 + np.exp(-eta01))
        p10 = 1 / (1 + np.exp(-eta10))

        prob = np.where(
            y_rep[:, t-1] == 0,
            p01,
            1 - p10
        )

        y_rep[:, t] = np.random.binomial(1, prob)

        neigh_sum = W @ y_rep[:, t]
        T_rep_iid[m, t-1] = neigh_sum.sum()

# ================================================================
# 8. POSTERIOR RANKING P-VALUES (TIME-WISE)
# ================================================================

p_time_bym = np.mean(T_rep_bym >= T_obs[None, :], axis=0)
p_time_iid = np.mean(T_rep_iid >= T_obs[None, :], axis=0)

def summarize(p):
    return {
        "mean": p.mean(),
        "sd": p.std(),
        "Pr(p < 0.05)": np.mean(p < 0.05),
        "Pr(p > 0.95)": np.mean(p > 0.95)
    }

print("\n===== Neighbor Snow PPC (Option 4) =====\n")
print("BYM:", summarize(p_time_bym))
print("IID:", summarize(p_time_iid))

# ================================================================
# 9. SAVE RESULTS
# ================================================================

np.savez(
    BASE_DIR / "ppc_neighbor_snow_time.npz",
    T_obs=T_obs,
    T_rep_bym=T_rep_bym,
    T_rep_iid=T_rep_iid,
    p_time_bym=p_time_bym,
    p_time_iid=p_time_iid
)

print("\nSaved to ppc_neighbor_snow_time.npz")


S = 1601, T = 2704
Posterior samples M = 1000

Running PPC (BYM)...


BYM neighbor PPC: 100%|██████████| 1000/1000 [10:26<00:00,  1.60it/s]



Running PPC (IID)...


IID neighbor PPC: 100%|██████████| 1000/1000 [08:17<00:00,  2.01it/s]


===== Neighbor Snow PPC (Option 4) =====

BYM: {'mean': 0.4999282278949316, 'sd': 0.4726784238465276, 'Pr(p < 0.05)': 0.4195338512763596, 'Pr(p > 0.95)': 0.42360340362560117}
IID: {'mean': 0.5083718091009989, 'sd': 0.472094193578727, 'Pr(p < 0.05)': 0.40769515353311137, 'Pr(p > 0.95)': 0.4335923048464669}

Saved to ppc_neighbor_snow_time.npz
